In [3]:
import os
os.listdir('/content')

['.config',
 'TS3.txt',
 'FS2.txt',
 'PS2.txt',
 'CP.txt',
 'TS1.txt',
 'profile.txt',
 'PS3.txt',
 'CE.txt',
 'PS4.txt',
 'SE.txt',
 'PS5.txt',
 'PS6.txt',
 'VS1.txt',
 'TS4.txt',
 'PS1.txt',
 'TS2.txt',
 'EPS1.txt',
 'description.txt',
 'documentation.txt',
 'FS1.txt',
 'sample_data']

In [4]:
import pandas as pd
import numpy as np

def extract_features(file):
    data = pd.read_csv(f'/content/{file}', header=None)

    features = pd.DataFrame()
    features['mean'] = data.mean(axis=1)
    features['std'] = data.std(axis=1)
    features['max'] = data.max(axis=1)
    features['min'] = data.min(axis=1)

    return features

In [12]:
import pandas as pd
import numpy as np

def extract_features(file):

    data = pd.read_csv(
        f'/content/{file}',
        header=None,
        sep=r'\s+',
        engine='python',
        on_bad_lines='skip'   # ⭐ KEY FIX
    )

    data = data.apply(pd.to_numeric, errors='coerce')

    features = pd.DataFrame()
    features['mean'] = data.mean(axis=1)
    features['std'] = data.std(axis=1)
    features['max'] = data.max(axis=1)
    features['min'] = data.min(axis=1)

    return features

In [13]:
all_features = pd.DataFrame()

for file in sensor_files:
    feat = extract_features(file)
    feat.columns = [file.replace('.txt','') + '_' + col for col in feat.columns]
    all_features = pd.concat([all_features, feat], axis=1)

all_features.head()

,PS1_mean,PS1_std,PS1_max,PS1_min,PS2_mean,PS2_std,PS2_max,PS2_min,PS3_mean,PS3_std,...,TS3_max,TS3_min,TS4_mean,TS4_std,TS4_max,TS4_min,VS1_mean,VS1_std,VS1_max,VS1_min
0,160.673492,13.939309,191.51,145.83,109.466914,47.114508,156.99,0.0,1.991475,0.945705,...,38.613,38.316,31.745250,1.116478,33.594,30.363,0.576950,0.027078,0.624,0.532
1,160.603320,14.118967,191.47,145.73,109.354890,47.045611,157.56,0.0,1.976234,0.941967,...,39.254,38.668,34.493867,0.435312,35.148,33.648,0.565850,0.027241,0.626,0.524
2,160.347720,14.192619,191.41,145.37,109.158845,46.992060,156.97,0.0,1.972224,0.943501,...,40.062,39.234,35.646150,0.293889,36.141,35.098,0.576533,0.036729,0.662,0.529
3,160.188088,14.227803,191.34,145.14,109.064807,46.972221,156.44,0.0,1.946575,0.935534,...,40.934,40.023,36.579467,0.262397,36.988,36.105,0.569267,0.033464,0.645,0.527
4,160.000472,14.276434,191.41,144.95,108.931434,46.874946,158.13,0.0,1.922707,0.930335,...,41.777,40.859,37.427900,0.239571,37.781,36.992,0.577367,0.033484,0.660,0.524


In [14]:
labels = pd.read_csv(
    '/content/profile.txt',
    header=None,
    sep=r'\s+',
    engine='python'
)

labels.head()

,0,1,2,3,4
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1
3,3,100,0,130,1
4,3,100,0,130,1


In [15]:
labels.shape

(2205, 5)

In [16]:
all_features.shape

(2237, 36)

In [24]:

print(all_features.isna().sum().sum())

1056


In [ ]:

all_features = all_features.fillna(all_features.mean())

In [25]:

all_features = all_features.dropna()

In [26]:
all_features['fault_label'] = labels[0]

all_features.head()

,PS1_mean,PS1_std,PS1_max,PS1_min,PS2_mean,PS2_std,PS2_max,PS2_min,PS3_mean,PS3_std,...,TS3_min,TS4_mean,TS4_std,TS4_max,TS4_min,VS1_mean,VS1_std,VS1_max,VS1_min,fault_label
0,160.673492,13.939309,191.51,145.83,109.466914,47.114508,156.99,0.0,1.991475,0.945705,...,38.316,31.745250,1.116478,33.594,30.363,0.576950,0.027078,0.624,0.532,3
1,160.603320,14.118967,191.47,145.73,109.354890,47.045611,157.56,0.0,1.976234,0.941967,...,38.668,34.493867,0.435312,35.148,33.648,0.565850,0.027241,0.626,0.524,3
2,160.347720,14.192619,191.41,145.37,109.158845,46.992060,156.97,0.0,1.972224,0.943501,...,39.234,35.646150,0.293889,36.141,35.098,0.576533,0.036729,0.662,0.529,3
3,160.188088,14.227803,191.34,145.14,109.064807,46.972221,156.44,0.0,1.946575,0.935534,...,40.023,36.579467,0.262397,36.988,36.105,0.569267,0.033464,0.645,0.527,3
4,160.000472,14.276434,191.41,144.95,108.931434,46.874946,158.13,0.0,1.922707,0.930335,...,40.859,37.427900,0.239571,37.781,36.992,0.577367,0.033484,0.660,0.524,3


In [27]:
all_features['fault_label'].value_counts()

,count
fault_label,
100,741
3,732
20,732


In [28]:
X = all_features.drop('fault_label', axis=1)
y = all_features['fault_label']

In [35]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y)

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [39]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("\n", name)
    print(classification_report(y_test, preds))


 Logistic Regression
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       152
           1       1.00      1.00      1.00       135
           2       1.00      0.99      1.00       154

    accuracy                           1.00       441
   macro avg       1.00      1.00      1.00       441
weighted avg       1.00      1.00      1.00       441


 SVM
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       152
           1       1.00      1.00      1.00       135
           2       1.00      0.98      0.99       154

    accuracy                           0.99       441
   macro avg       0.99      0.99      0.99       441
weighted avg       0.99      0.99      0.99       441


 Random Forest
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       152
           1       1.00      1.00      1.00       135
           2       1.00      0.

In [41]:
best_model = models["XGBoost"]

In [43]:
import joblib

joblib.dump(best_model, "industrial_fault_classifier.pkl")

['industrial_fault_classifier.pkl']

In [44]:
import os
os.listdir()

['.config',
 'TS3.txt',
 'FS2.txt',
 'PS2.txt',
 'CP.txt',
 'TS1.txt',
 'profile.txt',
 'PS3.txt',
 'CE.txt',
 'PS4.txt',
 'SE.txt',
 'PS5.txt',
 'PS6.txt',
 'VS1.txt',
 'TS4.txt',
 'PS1.txt',
 'industrial_fault_classifier.pkl',
 'TS2.txt',
 'EPS1.txt',
 'description.txt',
 'documentation.txt',
 'FS1.txt',
 'sample_data']